# Fine-tune YOLO26m trên Open Images V7 — 5 lớp rau củ quả

```
Input   : checkpoint gốc yolo26m.pt (huấn luyện trên COCO)
          + ~3.300 ảnh 5 lớp từ Open Images V7
Output  : checkpoint đã fine-tune
```

Notebook tự tải dữ liệu, không cần upload gì kèm theo.

## Trước khi chạy

| | |
|---|---|
| Accelerator | **GPU T4 x2** — Settings → Accelerator |
| Internet | **Bật** — Settings → Internet |
| Tải về | ~3,2 GB |
| Thời gian | 20–30 phút chuẩn bị dữ liệu, ~1,5 giờ huấn luyện |

Giới hạn của Kaggle là 12 giờ mỗi phiên nên toàn bộ nằm gọn trong một phiên.

### Phải là T4, không phải P100

PyTorch trên Kaggle (2.10.0+cu128) **không còn hỗ trợ P100**. Chọn nhầm sẽ gặp:

```
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible
with the current PyTorch installation.
The current PyTorch install supports sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120
```

P100 là kiến trúc Pascal năm 2016 (sm_60), đã bị các bản PyTorch mới bỏ hỗ trợ. T4 là
sm_75 nên chạy được.

Đổi lại còn nhanh hơn: T4 có nhân Tensor cho fp16 (~65 TFLOPS) trong khi P100 chỉ
khoảng 19 TFLOPS, mà notebook này bật `amp=True`.

Ô ở phần 0.1 kiểm tra điều này và dừng ngay nếu GPU không tương thích.

## Về Internet — đừng tắt giữa chừng

**Đổi thiết lập Internet sẽ khởi động lại kernel.** Không phải chỉ ngắt mạng — Kaggle
dựng lại phiên từ đầu, mọi biến mất sạch, tiến trình huấn luyện đang chạy bị giết.

Ba chỗ cần mạng, chỗ thứ ba dễ bị bỏ sót nhất:

| Phần | Cần mạng để làm gì |
|---|---|
| 0 | `pip install ultralytics` |
| 1–2 | Tải file annotation và ảnh |
| **4** | **Tải `yolo26m.pt`, và một model nano cho phép kiểm tra AMP** |

Khi `model.train()` bắt đầu, ultralytics tải thêm một checkpoint nano riêng để kiểm
tra AMP. Việc này xảy ra **sau** khi dữ liệu đã tải xong, nên tắt Internet ngay sau
phần 2 vì nghĩ "dữ liệu có rồi" sẽ làm hỏng phần huấn luyện.

Bật Internet và để nguyên suốt phiên.

## Vì sao không dùng thư viện fiftyone

Cách thông thường để lấy Open Images là qua `fiftyone`. Notebook này **không** dùng nó.

Trên Kaggle, `pip install fiftyone` nâng `pillow` lên bản mới và để lại trạng thái lẫn
lộn, khiến mọi thứ dùng PIL sau đó đều hỏng:

```
ImportError: cannot import name '_Ink' from 'PIL._typing'
```

Thay vào đó, notebook tải thẳng từ nguồn gốc của Google: file annotation dạng CSV, và
ảnh từ bucket S3 công khai. Chỉ cần `pandas` và `requests` — cả hai đều có sẵn trên
Kaggle, không cài đè lên gói nào.

Cách này còn đơn giản hơn: CSV của Open Images chứa toạ độ **đã chuẩn hoá sẵn** trong
khoảng [0, 1], đúng thứ YOLO cần, nên không phải đọc kích thước pixel của từng ảnh.

## 0. Thiết lập

In [ ]:
import concurrent.futures as cf
import json, os, shutil, subprocess, sys, time
from collections import Counter
from pathlib import Path

import pandas as pd
import requests

ON_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

DATA_DIR = WORK / "oi_data"
RUNS_DIR = WORK / "runs"
ARTIFACTS = WORK / "artifacts"
for d in (DATA_DIR, RUNS_DIR, ARTIFACTS):
    d.mkdir(parents=True, exist_ok=True)

# Thu tu nay quyet dinh chi so nhan trong file .txt cua YOLO.
CLASSES = ("apple", "banana", "broccoli", "carrot", "orange")
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# Ten lop trong Open Images viet hoa.
OI_NAMES = {"Apple": "apple", "Banana": "banana", "Broccoli": "broccoli",
            "Carrot": "carrot", "Orange": "orange"}

# Lop to tien trong cay phan cap cua Open Images - xem phan 2.
OI_ANCESTORS = {"Fruit", "Vegetable", "Food", "Produce"}

# Nguon du lieu chinh thuc cua Google.
URL_CLASSES = "https://storage.googleapis.com/openimages/v5/class-descriptions-boxable.csv"
URL_BBOX = {
    "train": "https://storage.googleapis.com/openimages/v6/oidv6-train-annotations-bbox.csv",
    "validation": "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv",
}
S3_IMAGE = "https://open-images-dataset.s3.amazonaws.com/{split}/{image_id}.jpg"

# Split `validation` chi co ~195 anh khop 5 lop - qua nho de chon checkpoint.
# Tap val duoc ghep them mot phan cat ra tu `train`. Xem phan 3.2.
VAL_FRACTION = 0.15
RANDOM_SEED = 42

print("Moi truong:", "Kaggle" if ON_KAGGLE else "local")
print("Thu muc lam viec:", WORK)

### 0.1 Cài ultralytics

Chỉ một gói. `pandas` và `requests` đã có sẵn trên Kaggle, và notebook không đụng
đến `torch`, `numpy`, hay `pillow`.

Nếu pip in ra vài dòng bắt đầu bằng `ERROR: pip's dependency resolver...` kèm tên
`google-adk`, `dopamine-rl`, `gradio` — đó là xung đột vốn có sẵn trong image của
Kaggle giữa các gói notebook này không dùng. Bỏ qua được.

In [ ]:
MODEL_NAME = "yolo26m.pt"     # doi thanh "yolo11m.pt" neu YOLO26 chua duoc ho tro

try:
    import ultralytics
    print("ultralytics co san:", ultralytics.__version__)
except ImportError:
    print("Dang cai ultralytics ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
    import ultralytics
    print("ultralytics:", ultralytics.__version__)

import torch
print("torch:", torch.__version__)

if not torch.cuda.is_available():
    raise SystemExit("KHONG THAY GPU - vao Settings -> Accelerator chon 'GPU T4 x2'")

props = torch.cuda.get_device_properties(0)
major, minor = torch.cuda.get_device_capability(0)
arch_list = [a for a in torch.cuda.get_arch_list() if a.startswith("sm_")]
print(f"GPU: {props.name}, {props.total_memory / 1024**3:.1f} GB, sm_{major}{minor}")
print("torch ho tro:", " ".join(arch_list))


def is_supported(major, minor, arch_list):
    """GPU chay duoc neu torch co cubin cung major version va minor khong lon hon.

    Khong the so khop chinh xac: cubin cua sm_86 chay duoc tren sm_89 vi cung
    major version 8. So khop chinh xac se bao loi nham cho nhung GPU hoan toan
    binh thuong.
    """
    for a in arch_list:
        digits = a[3:]
        if not digits.isdigit():
            continue
        if int(digits[:-1]) == major and int(digits[-1]) <= minor:
            return True
    return False


if not is_supported(major, minor, arch_list):
    raise SystemExit(
        f"\nGPU nay ({props.name}, sm_{major}{minor}) KHONG duoc ban torch nay ho tro.\n"
        f"torch {torch.__version__} chi ho tro: {' '.join(arch_list)}\n\n"
        f"Vao Settings -> Accelerator va chon 'GPU T4 x2'.\n"
        f"P100 la sm_60 va da bi cac ban torch moi bo ho tro."
    )

print("\nGPU tuong thich.")

### 0.2 Tải checkpoint ngay bây giờ

Ô dưới tải `yolo26m.pt` **trước** phần chuẩn bị dữ liệu, dù mãi tới phần 4 mới dùng đến.

Lý do: YOLO26 chỉ có từ ultralytics 8.4.117 trở lên. Nếu bản trên Kaggle cũ hơn, tên
checkpoint sẽ không tồn tại — và nếu chỉ phát hiện điều đó ở phần 4 thì bạn đã mất 30
phút chuẩn bị dữ liệu. Kiểm tra ở đây, lỗi lộ ra sau một phút.

Nếu ô này báo lỗi, đổi `MODEL_NAME` thành `"yolo11m.pt"` rồi chạy lại. YOLO11m cùng cỡ
(20,1 triệu tham số so với 21,9 triệu của YOLO26m) và phần còn lại chạy y nguyên.

In [ ]:
from ultralytics import YOLO

try:
    _probe = YOLO(MODEL_NAME)
    n_params = sum(p.numel() for p in _probe.model.parameters())
    print(f"{MODEL_NAME}: {n_params:,} tham so, {len(_probe.model.names)} lop (COCO)")
    del _probe
except Exception as exc:  # noqa: BLE001
    raise SystemExit(
        f"Khong tai duoc {MODEL_NAME}: {type(exc).__name__}: {exc}\n\n"
        f"ultralytics dang la ban {ultralytics.__version__}; YOLO26 can 8.4.117 tro len.\n"
        f'Sua MODEL_NAME thanh "yolo11m.pt" o o tren roi chay lai.'
    )

## 1. Tải annotation

Open Images phát hành nhãn dưới dạng CSV với các cột:

```
ImageID, Source, LabelName, Confidence, XMin, XMax, YMin, YMax,
IsOccluded, IsTruncated, IsGroupOf, IsDepiction, IsInside
```

`LabelName` là mã dạng `/m/014j1m` chứ không phải tên, nên cần bảng tra từ
`class-descriptions-boxable.csv`.

`XMin/XMax/YMin/YMax` **đã chuẩn hoá về [0, 1]** — đúng hệ toạ độ YOLO dùng, nên
không cần biết kích thước pixel của ảnh.

File của split `train` nặng **2.153 MB**. Notebook tải nó xuống đĩa trước rồi mới đọc,
thay vì đọc thẳng từ URL — vì một lần đứt mạng giữa chừng sẽ làm mất toàn bộ và phải
tải lại từ đầu. Hàm tải dùng header HTTP `Range` nên tiếp tục được từ chỗ dứt.

Sau khi tải, file được đọc theo từng khối 2 triệu dòng và chỉ giữ lại những dòng thuộc
5 lớp cần thiết, nên không bao giờ nạp cả file vào bộ nhớ. Đọc xong thì file CSV bị xoá
để trả lại dung lượng đĩa cho Kaggle.

In [ ]:
classes_df = pd.read_csv(URL_CLASSES, header=None, names=["LabelName", "DisplayName"])
mid_to_name = dict(zip(classes_df.LabelName, classes_df.DisplayName))
name_to_mid = {v: k for k, v in mid_to_name.items()}

WANT_MIDS = {name_to_mid[n]: n for n in OI_NAMES}
ANCESTOR_MIDS = {name_to_mid[n]: n for n in OI_ANCESTORS if n in name_to_mid}
KEEP_MIDS = set(WANT_MIDS) | set(ANCESTOR_MIDS)

print(f"{len(classes_df):,} lop boxable trong Open Images")
for mid, n in WANT_MIDS.items():
    print(f"   {n:<10} {mid}")
print("\nLop to tien de kiem tra:", sorted(ANCESTOR_MIDS.values()))

In [ ]:
USE_COLS = ["ImageID", "LabelName", "XMin", "XMax", "YMin", "YMax", "IsGroupOf"]
CHUNK = 2_000_000
CSV_DIR = WORK / "oi_csv"
CSV_DIR.mkdir(parents=True, exist_ok=True)


def download_resumable(url, dest: Path, retries=6):
    """Tai file lon, tiep tuc duoc tu cho dut thay vi bat dau lai.

    File cua split train nang 2,2 GB. Doc thang bang pd.read_csv(url) cung chay,
    nhung mot lan dut mang la mat toan bo va phai tai lai tu dau. Dung header
    HTTP Range de tiep tuc tu byte da co.
    """
    head = requests.head(url, timeout=30, allow_redirects=True)
    total = int(head.headers.get("Content-Length", 0))
    print(f"   {dest.name}: {total / 1024**2:,.0f} MB", flush=True)

    for attempt in range(retries):
        have = dest.stat().st_size if dest.exists() else 0
        if total and have >= total:
            print(f"   da co du {have / 1024**2:,.0f} MB", flush=True)
            return dest

        headers = {"Range": f"bytes={have}-"} if have else {}
        try:
            with requests.get(url, headers=headers, stream=True, timeout=120) as r:
                if r.status_code not in (200, 206):
                    raise RuntimeError(f"HTTP {r.status_code}")
                # 200 nghia la server bo qua Range -> phai ghi lai tu dau.
                mode = "ab" if (have and r.status_code == 206) else "wb"
                if mode == "wb":
                    have = 0
                t0, last = time.perf_counter(), have
                with dest.open(mode) as f:
                    for block in r.iter_content(1 << 20):
                        f.write(block)
                        have += len(block)
                        if have - last > 200 * 1024**2:
                            last = have
                            print(f"      {have / 1024**2:,.0f} / {total / 1024**2:,.0f} MB"
                                  f"  ({time.perf_counter() - t0:.0f}s)", flush=True)
            if not total or dest.stat().st_size >= total:
                return dest
        except Exception as exc:  # noqa: BLE001
            print(f"   lan {attempt + 1} that bai: {type(exc).__name__}: {exc}", flush=True)
            time.sleep(3 * (attempt + 1))

    raise RuntimeError(f"Khong tai duoc {url} sau {retries} lan")


def fetch_boxes(split):
    """Tai CSV ve dia roi doc theo khoi, chi giu dong thuoc 5 lop + lop to tien."""
    url = URL_BBOX[split]
    print(f"\n=== {split} ===", flush=True)
    path = download_resumable(url, CSV_DIR / Path(url).name)

    kept, rows_seen, t0 = [], 0, time.perf_counter()
    for i, chunk in enumerate(pd.read_csv(path, usecols=USE_COLS, chunksize=CHUNK)):
        rows_seen += len(chunk)
        kept.append(chunk[chunk.LabelName.isin(KEEP_MIDS)])
        print(f"   doc {rows_seen:>12,} dong, giu "
              f"{sum(len(k) for k in kept):>7,}  ({time.perf_counter() - t0:.0f}s)",
              flush=True)

    df = pd.concat(kept, ignore_index=True)
    df["name"] = df.LabelName.map({**WANT_MIDS, **ANCESTOR_MIDS})
    print(f"   xong: {rows_seen:,} dong -> {len(df):,} giu lai "
          f"({time.perf_counter() - t0:.0f}s)")

    # CSV goc khong con can, xoa de tra lai dung luong dia cho Kaggle.
    path.unlink(missing_ok=True)
    return df


boxes = {s: fetch_boxes(s) for s in ("validation", "train")}

In [ ]:
for split, df in boxes.items():
    ours = df[df.name.isin(OI_NAMES)]
    print(f"--- {split}: {len(ours):,} hop cua 5 lop tren {ours.ImageID.nunique():,} anh")
    for n in sorted(OI_NAMES):
        sub = ours[ours.name == n]
        print(f"    {n:<10} {len(sub):>6,} hop ({int(sub.IsGroupOf.sum()):>4} IsGroupOf) "
              f"tren {sub.ImageID.nunique():>5,} anh")

## 2. Lọc chất lượng

Ba bộ lọc, mỗi bộ có lý do cụ thể.

**Bỏ hộp `IsGroupOf`.** Open Images dùng một hộp duy nhất bao quanh cụm từ 5 vật thể
trở lên dính vào nhau. Đây là tương đương của `iscrowd` trong COCO. Một hộp bao cả rổ
táo không dạy được mô hình phát hiện từng quả. Chiếm khoảng **21% số hộp**.

**Loại ảnh có hộp tổ tiên mồ côi.** Đây là điểm dễ bỏ sót nhất.

Open Images gán nhãn theo cây phân cấp: `Apple` là con của `Fruit`, `Fruit` là con của
`Food`. Mỗi hộp mang đúng một nhãn — nhãn cụ thể nhất mà người gán nhãn xác định được.
Không chắc đó là quả gì thì hộp mang nhãn `Fruit`.

Hệ quả: một ảnh có 2 hộp `Apple` và 3 hộp `Fruit`, mà 3 hộp `Fruit` kia thực chất cũng
là táo, sẽ dạy mô hình rằng 3 vùng đó là **nền** — tức là dạy nó bỏ sót.

Cách xử lý không phải loại mọi ảnh có hộp tổ tiên (khoảng 31% số ảnh, quá phí), mà chỉ
loại khi hộp tổ tiên **không chồng lấn** hộp nào của 5 lớp. Một hộp `Fruit` phủ đúng
lên quả táo đã gắn nhãn `Apple` chỉ là cùng một vật thể mô tả ở hai mức, vô hại.

**Bỏ hộp suy biến** — chiều rộng hoặc cao bằng 0 sau khi cắt về [0, 1].

In [ ]:
DROP_ANCESTOR_IMAGES = True
ANCESTOR_IOU = 0.30      # tren nguong nay coi la cung mot vat the
MIN_REL_SIZE = 1e-4      # bo hop nho hon muc nay theo chieu bat ky


def iou_xyxy(a, b):
    ax0, ax1, ay0, ay1 = a
    bx0, bx1, by0, by1 = b
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    inter = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)
    union = (ax1 - ax0) * (ay1 - ay0) + (bx1 - bx0) * (by1 - by0) - inter
    return inter / union if union > 0 else 0.0


def build_records(df):
    """DataFrame -> {image_id: [(class, cx, cy, w, h), ...]} sau khi loc."""
    stats = Counter()
    records = {}

    for image_id, grp in df.groupby("ImageID", sort=False):
        # Anh chi co hop to tien (Fruit/Food/...) khong phai ung vien - bo qua
        # truoc khi dem, neu khong thi thong ke bi phong len rat nhieu.
        if not grp.name.isin(OI_NAMES).any():
            continue

        ours_xyxy, boxes_out = [], []
        for r in grp.itertuples():
            if r.name not in OI_NAMES:
                continue
            if r.IsGroupOf == 1:
                stats["bo_hop_IsGroupOf"] += 1
                continue
            x0, x1 = max(0.0, r.XMin), min(1.0, r.XMax)
            y0, y1 = max(0.0, r.YMin), min(1.0, r.YMax)
            if x1 - x0 < MIN_REL_SIZE or y1 - y0 < MIN_REL_SIZE:
                stats["bo_hop_suy_bien"] += 1
                continue
            ours_xyxy.append((x0, x1, y0, y1))
            boxes_out.append((OI_NAMES[r.name],
                              (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0))

        if not boxes_out:
            stats["bo_anh_khong_nhan"] += 1
            continue

        if DROP_ANCESTOR_IMAGES:
            orphan = False
            for r in grp.itertuples():
                if r.name not in OI_ANCESTORS:
                    continue
                anc = (r.XMin, r.XMax, r.YMin, r.YMax)
                if max((iou_xyxy(anc, o) for o in ours_xyxy), default=0.0) < ANCESTOR_IOU:
                    orphan = True
                    break
            if orphan:
                stats["bo_anh_to_tien_mo_coi"] += 1
                continue

        records[image_id] = boxes_out

    # Dem tren anh co it nhat mot hop cua 5 lop. `df` con chua ca anh chi co hop
    # to tien (Fruit/Food/...), va nhung anh do khong bao gio la ung vien.
    stats["anh_vao"] = df[df.name.isin(OI_NAMES)].ImageID.nunique()
    stats["anh_giu_lai"] = len(records)
    stats["instance_giu_lai"] = sum(len(v) for v in records.values())
    return records, stats


filtered = {}
for split, df in boxes.items():
    recs, st = build_records(df)
    filtered[split] = recs
    print(f"--- {split}")
    for k in ("anh_vao", "bo_hop_IsGroupOf", "bo_hop_suy_bien",
              "bo_anh_to_tien_mo_coi", "bo_anh_khong_nhan",
              "anh_giu_lai", "instance_giu_lai"):
        print(f"    {k:<24} {st[k]:>7,}")

## 3. Tải ảnh và ghi dataset

### 3.1 Chia tập val

Split `validation` của Open Images chỉ còn khoảng 80 ảnh sau khi lọc, quá nhỏ để chọn
checkpoint — `patience` sẽ phản ứng với nhiễu chứ không phải tiến bộ thật, và `best.pt`
mà ultralytics giữ lại gần như là chọn ngẫu nhiên.

Nên tập val được ghép từ hai nguồn: chính split `validation`, cộng thêm 15% cắt ra từ
`train`. Phép cắt phân tầng theo **tập lớp có trong ảnh** chứ không theo một lớp chính,
để ảnh nhiều lớp không dồn hết về một bên.

In [ ]:
import random


def stratified_split(items, fraction, seed):
    """Cat `fraction` lam val, phan tang theo tap lop co trong anh."""
    buckets = {}
    for image_id, boxes_out in items:
        buckets.setdefault(tuple(sorted({b[0] for b in boxes_out})), []).append(
            (image_id, boxes_out))

    rng = random.Random(seed)
    train, val = [], []
    for sig in sorted(buckets):
        group = sorted(buckets[sig], key=lambda t: t[0])
        rng.shuffle(group)
        n_val = round(len(group) * fraction)
        n_val = min(n_val, len(group) - 1) if len(group) > 1 else 0
        val.extend(group[:n_val])
        train.extend(group[n_val:])
    return train, val


train_items, carved = stratified_split(
    list(filtered["train"].items()), VAL_FRACTION, RANDOM_SEED)

# Chot chan: mot anh khong duoc nam ca o train lan val. Hai split cua Open Images
# vo nguyen tac la roi nhau nen dieu nay khong xay ra, nhung neu xay ra thi day la
# ro ri du lieu am tham - mo hinh duoc cham diem tren chinh anh no da hoc.
train_ids, seen = {i for i, _ in train_items}, set()
val_items = []
for image_id, bs in carved + list(filtered["validation"].items()):
    if image_id in train_ids or image_id in seen:
        continue
    seen.add(image_id)
    val_items.append((image_id, bs))

# Anh cua split nao thi tai tu duong dan S3 cua split do.
origin = {i: "train" for i, _ in filtered["train"].items()}
origin.update({i: "validation" for i, _ in filtered["validation"].items()})

plan = {"train": train_items, "val": val_items}
print(f"train  {len(filtered['train']):>5,} -> {len(train_items):>5,} "
      f"(cat {len(carved):,} sang val)")
print(f"val    {len(filtered['validation']):>5,} + {len(carved):,} = {len(val_items):>5,}")

In [ ]:
# Can can lop. Lop nao qua it thi ket qua tren lop do se khong dang tin.
print(f"{'lop':<12}{'train':>10}{'val':>10}{'tong':>10}")
print("-" * 42)
totals = Counter()
for c in CLASSES:
    row = f"{c:<12}"
    for split in ("train", "val"):
        n = sum(1 for _, bs in plan[split] for b in bs if b[0] == c)
        totals[c] += n
        row += f"{n:>10,}"
    print(row + f"{totals[c]:>10,}")
print("-" * 42)
print(f"{'anh':<12}{len(plan['train']):>10,}{len(plan['val']):>10,}"
      f"{len(plan['train']) + len(plan['val']):>10,}")

n_val = len(plan["val"])
print(f"\nTap val: {n_val:,} anh, {sum(len(b) for _, b in plan['val']):,} instance")
if n_val < 300:
    print("Van kha nho - can nhac tang VAL_FRACTION o phan 0.")

### 3.2 Tải ảnh

Ảnh nằm trong bucket S3 công khai `open-images-dataset`, tải theo `ImageID`. Chỉ tải
đúng những ảnh còn lại sau khi lọc.

Tải song song bằng nhiều luồng vì mỗi ảnh chỉ khoảng 300 KB — thời gian chủ yếu là chờ
mạng chứ không phải băng thông.

In [ ]:
DOWNLOAD_WORKERS = 32
RETRIES = 3


def fetch_one(args):
    image_id, split_dir, dest = args
    if dest.exists() and dest.stat().st_size > 0:
        return image_id, True
    url = S3_IMAGE.format(split=split_dir, image_id=image_id)
    for attempt in range(RETRIES):
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200 and r.content:
                dest.write_bytes(r.content)
                return image_id, True
        except Exception:  # noqa: BLE001
            pass
        time.sleep(0.5 * (attempt + 1))
    return image_id, False


downloaded = {}
for split, items in plan.items():
    img_dir = DATA_DIR / "images" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    jobs = [(i, origin[i], img_dir / f"{i}.jpg") for i, _ in items]

    print(f"\n=== tai {len(jobs):,} anh cho '{split}' ===", flush=True)
    ok, t0 = set(), time.perf_counter()
    with cf.ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
        for n, (image_id, good) in enumerate(pool.map(fetch_one, jobs), 1):
            if good:
                ok.add(image_id)
            if n % 500 == 0 or n == len(jobs):
                print(f"   {n:>6,}/{len(jobs):,}  thanh cong {len(ok):>6,}  "
                      f"({time.perf_counter() - t0:.0f}s)", flush=True)

    downloaded[split] = ok
    missing = len(jobs) - len(ok)
    print(f"   {split}: {len(ok):,} anh, {missing:,} that bai")

### 3.3 Ghi nhãn YOLO

Ultralytics suy ra thư mục nhãn bằng cách thay `images` thành `labels` trong đường dẫn,
nên hai cây thư mục phải song song đúng quy ước đó.

Toạ độ trong CSV đã ở dạng chuẩn hoá nên chỉ cần đổi từ `xyxy` sang `cxcywh` — không có
phép nhân chia nào theo kích thước ảnh, tức là không có chỗ để sai.

In [ ]:
def write_labels(plan, downloaded, out_dir: Path):
    written = {}
    for split, items in plan.items():
        lbl_dir = out_dir / "labels" / split
        lbl_dir.mkdir(parents=True, exist_ok=True)
        n = 0
        for image_id, boxes_out in items:
            if image_id not in downloaded[split]:
                continue          # anh tai that bai -> bo qua ca nhan
            lines = [f"{CLASS_TO_IDX[c]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"
                     for c, cx, cy, w, h in boxes_out]
            (lbl_dir / f"{image_id}.txt").write_text("\n".join(lines) + "\n",
                                                     encoding="utf-8")
            n += 1
        written[split] = n
        print(f"  {split:<6} {n:>6,} nhan")
    return written


write_labels(plan, downloaded, DATA_DIR)

# `path` phai tuyet doi: ultralytics giai duong dan split tuong doi voi no.
(DATA_DIR / "data.yaml").write_text(
    f"path: {DATA_DIR.resolve().as_posix()}\n"
    "train: images/train\nval: images/val\n\n"
    f"nc: {len(CLASSES)}\nnames:\n"
    + "".join(f"  {i}: {c}\n" for i, c in enumerate(CLASSES)),
    encoding="utf-8")
print()
print((DATA_DIR / "data.yaml").read_text(encoding="utf-8"))

In [ ]:
# Chan lai truoc khi ton gio GPU neu du lieu ghi ra khong dung.
ok = True
for split in ("train", "val"):
    imgs = {p.stem for p in (DATA_DIR / "images" / split).glob("*.jpg")}
    lbls = {p.stem for p in (DATA_DIR / "labels" / split).glob("*.txt")}
    orphan_img = imgs - lbls
    orphan_lbl = lbls - imgs
    empty = [p.name for p in (DATA_DIR / "labels" / split).glob("*.txt")
             if not p.read_text(encoding="utf-8").strip()]
    good = not orphan_lbl and not empty and len(lbls) > 0
    ok &= good
    print(f"  {split:<6} anh={len(imgs):<6} nhan={len(lbls):<6} "
          f"anh_thua={len(orphan_img):<4} nhan_thua={len(orphan_lbl):<4} "
          f"rong={len(empty):<3} {'OK' if good else 'LOI'}")

assert ok, "Du lieu ghi ra khong khop - dung lai truoc khi huan luyen"
print("\nDu lieu san sang.")

## 4. Huấn luyện YOLO26m

Tham số theo [công thức fine-tune chính thức của YOLO26](https://docs.ultralytics.com/guides/yolo26-training-recipe).

Vài điểm cần biết:

- `optimizer="auto"` tự chọn **MuSGD** khi số bước vượt 10.000, ngược lại chọn AdamW.
  Với khoảng 2.700 ảnh train và batch 16 thì mỗi epoch chừng 170 bước, nên 60 epoch đạt
  ~10.200 bước — **sát ngay ngưỡng**. Giảm `EPOCHS` xuống dưới 59 sẽ khiến ultralytics
  chuyển sang AdamW và kết quả có thể khác hẳn. Nếu muốn cố định, đặt thẳng
  `optimizer="MuSGD"` hoặc `"AdamW"`.
- Không được truyền các tham số nội bộ `muon_w`, `sgd_w`, `cls_w`, `o2m`, `topk` —
  chúng chỉ tồn tại trong checkpoint và sẽ gây lỗi.
- `batch=16` là **ngoại suy chưa kiểm chứng**: YOLO11s ở batch 16 chiếm 4,24 GB trên
  RTX 4060, YOLO26m nặng khoảng ba lần về FLOPs nên ước 10–11 GB; T4 có 16 GB.
  **Chạy `EPOCHS = 1` một lần trước** để xem VRAM và tốc độ thật rồi mới chạy đủ.
- `workers=8` vì Kaggle chạy Linux, không dùng `spawn` như Windows.
- `device=0` dùng một GPU T4. Chọn `GPU T4 x2` để có T4 đầu tiên; dùng cả hai
  (`device=[0, 1]`) cần DDP, mà DDP trong notebook hay trục trặc và với ~2.700 ảnh
  thì phần chi phí đồng bộ ăn gần hết phần lợi.
- `save_period=10` để phiên bị ngắt vẫn còn checkpoint dùng được.

In [ ]:
# MODEL_NAME da duoc dat va kiem tra o phan 0.2.
RUN_NAME = "yolo26m_oi"
EPOCHS = 60                  # dat 1 cho lan chay thu dau tien

# --- Augmentation ---
# Viet tuong minh thay vi de mac dinh an, de chinh duoc. Cac gia tri duoi day dung
# bang mac dinh cua ultralytics. Tai lieu YOLO26 khuyen *giam* augmentation cho tap
# duoi 1.000 anh chu khong tang; tap nay khoang 2.700 anh nam giua hai nguong, nen
# bat dau bang mac dinh la lua chon an toan.
AUG = dict(
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=0.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0,
    fliplr=0.5, flipud=0.0,
    mosaic=1.0, mixup=0.0, copy_paste=0.0,
)

# Dat True de tang cuong manh hon. Rau cu qua khong co huong co dinh nen xoay nhe la
# hop ly; mixup va copy_paste thuong giup khi du lieu it.
STRONGER_AUG = False
if STRONGER_AUG:
    AUG |= dict(degrees=5.0, mixup=0.1, copy_paste=0.1)

TRAIN_ARGS = dict(
    data=str(DATA_DIR / "data.yaml"),
    epochs=EPOCHS, imgsz=640, batch=16,
    optimizer="auto", lr0=0.001, lrf=0.01,
    momentum=0.948, weight_decay=0.00027, warmup_epochs=0.99,
    cos_lr=True, amp=True, workers=8,
    patience=20, close_mosaic=10, save_period=10,
    seed=RANDOM_SEED, project=str(RUNS_DIR), name=RUN_NAME,
    exist_ok=True, val=True, plots=True,
    device=0 if torch.cuda.is_available() else "cpu",
    **AUG,
)

for k, v in TRAIN_ARGS.items():
    print(f"  {k:<16} {v}")

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)
print("Tham so:", f"{sum(p.numel() for p in model.model.parameters()):,}")
print("Checkpoint goc co", len(model.model.names), "lop (COCO)")

t0 = time.perf_counter()
results = model.train(**TRAIN_ARGS)
train_minutes = (time.perf_counter() - t0) / 60
print(f"\nHuan luyen xong sau {train_minutes:.1f} phut")

if torch.cuda.is_available():
    print(f"VRAM dinh: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

## 5. Lưu checkpoint

Trên Kaggle, `/kaggle/working` được giữ lại khi bạn bấm **Save Version**. Tải thư mục
`artifacts/` về sau khi phiên chạy xong.

In [ ]:
run_dir = RUNS_DIR / RUN_NAME
pack = ARTIFACTS / RUN_NAME
pack.mkdir(parents=True, exist_ok=True)

for rel in ("weights/best.pt", "weights/last.pt", "results.csv", "args.yaml"):
    src = run_dir / rel
    if src.is_file():
        shutil.copy2(src, pack / Path(rel).name)
        print(f"  {Path(rel).name:<14} {src.stat().st_size / 1024**2:>7.1f} MB")

(pack / "summary.json").write_text(json.dumps({
    "model": MODEL_NAME,
    "dataset": "open-images-v7, 5 lop, tai truc tiep tu CSV + S3",
    "epochs_configured": EPOCHS,
    "train_minutes": round(train_minutes, 1),
    "images": {s: len(downloaded[s]) for s in ("train", "val")},
    "instances": {s: sum(len(b) for i, b in plan[s] if i in downloaded[s])
                  for s in ("train", "val")},
    "train_args": {k: str(v) for k, v in TRAIN_ARGS.items()},
}, indent=1), encoding="utf-8")

print(f"\nCheckpoint da fine-tune: {pack / 'best.pt'}")

### Nếu phiên Kaggle bị ngắt

Kaggle giới hạn 12 giờ mỗi phiên, và phần lưu checkpoint ở trên chỉ chạy nếu tiến trình
sống tới cuối. Một phiên bị giết sẽ để lại `runs/yolo26m_oi/weights/last.pt` nhưng
không có thư mục `artifacts/`.

Chạy tiếp bằng cách thêm `resume=True` vào `TRAIN_ARGS`, giữ **nguyên** `project` và
`name`. Lưu ý ultralytics khôi phục `args.yaml` gốc kể cả `epochs`, nên đổi `EPOCHS`
lúc resume sẽ không có tác dụng — phải đặt đúng ngay từ đầu.